# Dataset Preprocessing following Weytjens and De Weerdt.

In [ ]:
import time
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.log.exporter.xes import exporter as xes_exporter
from pm4py.objects.conversion.log import converter
import os

###############################################################################
# Utility: print basic dataset info
###############################################################################
def print_dataset_info(df, msg="Dataset info"):
    n_cases = df["case:concept:name"].nunique()
    n_rows = len(df)
    print(f"{msg}: {n_cases} cases, {n_rows} rows")

###############################################################################
# Steps 1 & 2: Filtering by start and end dates
###############################################################################
def start_from_date(dataset, start_date):
    print("\n=== Remove Cases that start before", start_date, "===")
    case_starts_df = dataset.groupby("case:concept:name")["time:timestamp"].min().reset_index()
    case_starts_df["date"] = case_starts_df["time:timestamp"].dt.to_period("M")
    cases_after = case_starts_df[case_starts_df["date"].astype(str) >= start_date]["case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_after)].reset_index(drop=True)
    print_dataset_info(dataset, f"After start_from_date >= {start_date}")
    return dataset

def end_before_date(dataset, end_date):
    print("\n=== Remove Cases that end after", end_date, "===")
    case_stops_df = dataset.groupby("case:concept:name")["time:timestamp"].max().reset_index()
    case_stops_df["date"] = case_stops_df["time:timestamp"].dt.to_period("M")
    cases_before = case_stops_df[case_stops_df["date"].astype(str) <= end_date]["case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_before)].reset_index(drop=True)
    print_dataset_info(dataset, f"After end_before_date <= {end_date}")
    return dataset

###############################################################################
# Step 3: Remove long cases and debias dataset endpoint
###############################################################################
def limited_duration(dataset, max_duration):
    print("\n=== Remove Long Cases (<= {} days) ===".format(max_duration))
    agg_dict = {"time:timestamp": ["min", "max"]}
    duration_df = dataset.groupby("case:concept:name").agg(agg_dict).reset_index()
    duration_df["duration"] = (
        duration_df[("time:timestamp", "max")] - duration_df[("time:timestamp", "min")]
    ).dt.total_seconds() / (24 * 60 * 60)
    condition_1 = duration_df["duration"] <= max_duration * 1.00000000001
    cases_retained_1 = duration_df[condition_1]["case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_retained_1)].reset_index(drop=True)
    latest_start = dataset["time:timestamp"].max() - pd.Timedelta(max_duration, unit="D")
    new_min_df = dataset.groupby("case:concept:name")["time:timestamp"].min().reset_index()
    new_min_df.columns = ["case:concept:name", "min_timestamp"]
    condition_2 = new_min_df["min_timestamp"] <= latest_start
    cases_retained_2 = new_min_df[condition_2]["case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_retained_2)].reset_index(drop=True)
    print_dataset_info(dataset, f"After limited_duration:")
    return dataset, latest_start

###############################################################################
# Pre-processing: timestamp conversion, remain_time calculation, filtering and duplicate removal
###############################################################################
def preprocess_dataset(dataset, start_date, end_date, max_duration):
    print_dataset_info(dataset, "Initial dataset")
    dataset["time:timestamp"] = pd.to_datetime(dataset["time:timestamp"], utc=True)
    dataset["time:timestamp"] = dataset["time:timestamp"].dt.tz_convert(None)
    dataset["remain_time"] = (
        dataset.groupby("case:concept:name")["time:timestamp"].transform("max")
        - dataset["time:timestamp"]
    )
    dataset["remain_time"] = dataset["remain_time"].dt.total_seconds() / (24 * 60 * 60)
    if start_date:
        dataset = start_from_date(dataset, start_date)
    if end_date:
        dataset = end_before_date(dataset, end_date)
    print("\n=== Remove Duplicates ===")
    dataset.drop_duplicates(inplace=True)
    print_dataset_info(dataset, "After dropping duplicates")
    dataset_short, latest_start = limited_duration(dataset, max_duration)
    return dataset_short, latest_start

###############################################################################
# Step 4a: Temporal split into train and test sets
###############################################################################
def trainTestSplit(df, test_len, latest_start, targets, remove_missing_prefix_cases=False):
    print("\n=== trainTestSplit Step ===")
    print_dataset_info(df, "Before trainTestSplit")
    print(f"Removing the Cases with missing prefixes = {remove_missing_prefix_cases}")
    if isinstance(targets, str):
        targets = [targets]
    case_starts_df = df.groupby("case:concept:name")["time:timestamp"].min()
    case_stops_df  = df.groupby("case:concept:name")["time:timestamp"].max().to_frame()
    sorted_start_values = np.sort(case_starts_df.values)
    first_test_case_nr = int(len(sorted_start_values) * (1 - test_len))
    first_test_start_time = sorted_start_values[first_test_case_nr]
    naive_train_ids = case_stops_df[case_stops_df["time:timestamp"] < first_test_start_time].index
    naive_test_ids  = case_stops_df[case_stops_df["time:timestamp"] >= first_test_start_time].index
    print(f"\nSplitting dataset with temporal strict rules -> Train cases: {len(naive_train_ids)}, Test cases: {len(naive_test_ids)}")
    bridging_mask = (case_starts_df < first_test_start_time) & (case_stops_df["time:timestamp"] >= first_test_start_time)
    print(f"Number of bridging cases: {bridging_mask.sum()}")
    df_test_all = df[df["case:concept:name"].isin(naive_test_ids)].copy().reset_index(drop=True)
    case_stops_test = df_test_all.groupby("case:concept:name")["time:timestamp"].max()
    missing_long_prefixes = case_stops_test[case_stops_test > latest_start].index
    case_starts_test = df_test_all.groupby("case:concept:name")["time:timestamp"].min()
    missing_short_prefixes = case_starts_test[case_starts_test < first_test_start_time].index
    if remove_missing_prefix_cases:
        to_remove = set(missing_long_prefixes) | set(missing_short_prefixes)
        print(f"Removing {len(to_remove)} test cases with missing prefixes.")
        df_test_all = df_test_all[~df_test_all["case:concept:name"].isin(to_remove)].copy().reset_index(drop=True)
    df_test = df_test_all[df_test_all["time:timestamp"] <= latest_start].copy().reset_index(drop=True)
    df_test.loc[df_test["time:timestamp"] < first_test_start_time, targets] = np.nan
    df_train = df[df["case:concept:name"].isin(naive_train_ids)].copy().reset_index(drop=True)
    print_dataset_info(df_train, "Final training set")
    print_dataset_info(df_test, "Final test set")
    return df_train, df_test

###############################################################################
# Step 4b: Temporal split into train, test and calibration sets
###############################################################################
def trainTestCalibrationSplit(df, test_share, cons_share, latest_start, targets, remove_missing_prefix_cases=False):
    print("\n=== Dataset Split (Training set: 60%, Test set: 20%, Calibration set: 20%) ===")
    print(f"Removing the Cases with missing prefixes = {remove_missing_prefix_cases}")
    if isinstance(targets, str):
        targets = [targets]
    case_starts_df = df.groupby("case:concept:name")["time:timestamp"].min()
    case_stops_df = df.groupby("case:concept:name")["time:timestamp"].max().to_frame()
    sorted_starts = np.sort(case_starts_df.values)
    total_cases = len(sorted_starts)
    training_share = 1 - test_share - cons_share
    first_threshold_index = int(total_cases * training_share)
    second_threshold_index = int(total_cases * (training_share + test_share))
    first_split_time = sorted_starts[first_threshold_index]
    second_split_time = sorted_starts[second_threshold_index]
    naive_train_ids = case_stops_df[case_stops_df["time:timestamp"] < first_split_time].index
    naive_test_ids = case_stops_df[(case_stops_df["time:timestamp"] >= first_split_time) &
                                   (case_stops_df["time:timestamp"] < second_split_time)].index
    naive_cons_ids = case_stops_df[case_stops_df["time:timestamp"] >= second_split_time].index
    print(f"Number of Cases for each set -> Train: {len(naive_train_ids)}, Test: {len(naive_test_ids)}, Calibration: {len(naive_cons_ids)}")
    df_test_all = df[df["case:concept:name"].isin(naive_test_ids)].copy().reset_index(drop=True)
    df_cons_all = df[df["case:concept:name"].isin(naive_cons_ids)].copy().reset_index(drop=True)
    case_stops_test = df_test_all.groupby("case:concept:name")["time:timestamp"].max()
    missing_long_prefixes_test = case_stops_test[case_stops_test > latest_start].index
    case_stops_cons = df_cons_all.groupby("case:concept:name")["time:timestamp"].max()
    missing_long_prefixes_cons = case_stops_cons[case_stops_cons > latest_start].index
    case_starts_test = df_test_all.groupby("case:concept:name")["time:timestamp"].min()
    missing_short_prefixes_test = case_starts_test[case_starts_test < first_split_time].index
    case_starts_cons = df_cons_all.groupby("case:concept:name")["time:timestamp"].min()
    missing_short_prefixes_cons = case_starts_cons[case_starts_cons < second_split_time].index
    if remove_missing_prefix_cases:
        to_remove_test = set(missing_long_prefixes_test) | set(missing_short_prefixes_test)
        to_remove_cons = set(missing_long_prefixes_cons) | set(missing_short_prefixes_cons)
        print(f"\nRemoving {len(to_remove_test)} test cases and {len(to_remove_cons)} calibration cases with missing prefixes.")
        df_test_all = df_test_all[~df_test_all["case:concept:name"].isin(to_remove_test)].copy().reset_index(drop=True)
        df_cons_all = df_cons_all[~df_cons_all["case:concept:name"].isin(to_remove_cons)].copy().reset_index(drop=True)
    df_test = df_test_all[df_test_all["time:timestamp"] <= latest_start].copy().reset_index(drop=True)
    df_cons = df_cons_all[df_cons_all["time:timestamp"] <= latest_start].copy().reset_index(drop=True)
    df_test.loc[df_test["time:timestamp"] < first_split_time, targets] = np.nan
    df_cons.loc[df_cons["time:timestamp"] < second_split_time, targets] = np.nan
    df_train = df[df["case:concept:name"].isin(naive_train_ids)].copy().reset_index(drop=True)
    print_dataset_info(df_train, "Final training set")
    print_dataset_info(df_test, "Final test set")
    print_dataset_info(df_cons, "Final calibration set")
    return df_train, df_test, df_cons

###############################################################################
# Classification target
###############################################################################
def createClassificationTargets(df, col, keywords_dict):
    print("\n=== createClassificationTargets Step ===")
    print_dataset_info(df, "Before creating classification targets")
    def fill_targets(grp):
        for target, keywords in keywords_dict.items():
            grp[target] = any(x in grp[col].values for x in keywords)
        return grp
    df = df.groupby("case:concept:name").apply(fill_targets)
    print_dataset_info(df, "After creating classification targets")
    return df

###############################################################################
# Add end_timestamp column
###############################################################################
def add_end_timestamp(df):
    df = df.sort_values(["case:concept:name", "time:timestamp"])
    df["end_timestamp"] = df.groupby("case:concept:name")["time:timestamp"].shift(-1)
    return df

###############################################################################
# Helper function: unique_preserve_order
###############################################################################
def unique_preserve_order(seq):
    """
    Returns a list containing the values of seq in the order of first occurrence, removing duplicates.
    """
    seen = set()
    result = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

###############################################################################
# Feature Generation: ACCORPA TUTTE LE RIPETIZIONI DELLA STESSA ATTIVITÀ
###############################################################################
def generate_features(df, remove_incomplete=False):
    """
    For each case and activity, merge ALL events
    (even if there are other activities in between).
    - occurrence_id: an integer assigned based on the order of appearance (sorted by start_time) of each activity in the case.
    - start_time: the minimum timestamp among all events of that activity. 
    - end_time: the maximum timestamp.
    - lifecycle_list, org:resource_list: a union of all encountered values, without duplicates, maintaining the order of first appearance.
    """
    df = df.sort_values(["case:concept:name", "time:timestamp"]).reset_index(drop=True)
    
    # Raggruppiamo unicamente per (caso, attività), ignorando la consecutività
    grouped = df.groupby(["case:concept:name", "concept:name"], sort=False)
    
    temp_records = []
    for (case_id, activity), subdf in grouped:
        subdf = subdf.sort_values("time:timestamp")
        start_time = subdf["time:timestamp"].min()
        end_time = subdf["time:timestamp"].max()
        
        # Aggrega i lifecycle e i resource preservando l'ordine e senza duplicati
        lifecycles = unique_preserve_order(subdf["lifecycle:transition"])
        resources = unique_preserve_order(subdf["org:resource"])
        
        temp_records.append({
            "case:concept:name": case_id,
            "concept:name": activity,
            "start_time": start_time,
            "end_time": end_time,
            "lifecycle_list": lifecycles,
            "org:resource_list": resources
        })
    
    # Now we have a row for each (case, activity)  
    # We assign occurrence_id based on the order of start_time within each case  
    temp_df = pd.DataFrame(temp_records)
    
    final_records = []
    for case_id, case_df in temp_df.groupby("case:concept:name", sort=False):
        case_df = case_df.sort_values("start_time", ascending=True)
        case_df = case_df.reset_index(drop=True)
        # Assegna occurrence_id come un contatore intero, partendo da 1
        case_df["occurrence_id"] = range(1, len(case_df) + 1)
        final_records.append(case_df)
    
    final_df = pd.concat(final_records, ignore_index=True)
    
    if remove_incomplete:
        final_df = final_df[final_df["start_time"] < final_df["end_time"]].reset_index(drop=True)
    
    # Numero di eventi originali meno numero di righe finali
    events_eliminated = len(df) - len(final_df)
    return final_df, events_eliminated

###############################################################################
# Analysis: compare original events vs generated features
###############################################################################
def analyze_features(original_df, features_df):
    total_original = len(original_df)
    total_features = len(features_df)
    total_cases = original_df["case:concept:name"].nunique()
    print(f"\nTotal original events: {total_original}")
    print(f"Total feature occurrences: {total_features}")
    print(f"Total cases: {total_cases}")
    orig_counts = original_df.groupby("case:concept:name").size().rename("original_event_count")
    feat_counts = features_df.groupby("case:concept:name").size().rename("feature_occurrence_count")
    summary = pd.concat([orig_counts, feat_counts], axis=1).fillna(0).astype(int)
    summary["eliminated"] = summary["original_event_count"] - summary["feature_occurrence_count"]
    total_eliminated = summary["eliminated"].sum()
    print(f"\nOverall, {total_eliminated} event rows were reduced into feature occurrences across {total_cases} cases.")

###############################################################################
# Main function for benchmark creation and feature generation
###############################################################################
def remainTimeOrClassifBenchmark(
    dataset, path, file_name, start_date, end_date, max_days, test_len_share,
    output_type="csv", keywords_dict=None,
    remove_missing_prefix_cases=False, calibration_share=None, apply_calibration=True
):
    # Pre-process the dataset
    dataset_proc, latest_start = preprocess_dataset(dataset, start_date, end_date, max_days)
    
    targets = "remain_time"
    if keywords_dict:
        dataset_proc["classif_target"] = dataset_proc["concept:name"]
        dataset_proc = createClassificationTargets(dataset_proc, "classif_target", keywords_dict)
        dataset_proc.drop(columns=["classif_target"], inplace=True)
        targets = list(keywords_dict.keys()) + ["remain_time"]
    
    # Splitting: either train/test or train/test/calibration
    if calibration_share is not None:
        df_train, df_test, df_cons = trainTestCalibrationSplit(
            dataset_proc, test_len_share, calibration_share, latest_start, targets,
            remove_missing_prefix_cases=remove_missing_prefix_cases
        )
    else:
        df_train, df_test = trainTestSplit(
            dataset_proc, test_len_share, latest_start, targets,
            remove_missing_prefix_cases=remove_missing_prefix_cases
        )
        df_cons = None

    df_train = add_end_timestamp(df_train)
    df_test  = add_end_timestamp(df_test)
    if df_cons is not None:
        df_cons = add_end_timestamp(df_cons)
    
    folder_name = "remove_partial" if remove_missing_prefix_cases else "keep_partial"
    out_folder = os.path.join(path, folder_name)
    os.makedirs(out_folder, exist_ok=True)
    
    print("\n=== Feature Generation ===")
    final_train, removed_train = generate_features(df_train)
    # Normalizza eventuali "O_Sent (mail ...)" in "O_Sent"
    final_train["concept:name"] = final_train["concept:name"].str.replace(r"^O_Sent\s*\(.*\)", "O_Sent", regex=True)
    final_train = final_train.sort_values(["case:concept:name", "occurrence_id"], ascending=[True, True]).reset_index(drop=True)
    print("\n1. Training Set Final Features:")
    analyze_features(df_train, final_train)
    final_train.to_csv(os.path.join(out_folder, "train_set.csv"), index=False)
    
    final_test, removed_test = generate_features(df_test)
    final_test["concept:name"] = final_test["concept:name"].str.replace(r"^O_Sent\s*\(.*\)", "O_Sent", regex=True)
    final_test = final_test.sort_values(["case:concept:name", "occurrence_id"], ascending=[True, True]).reset_index(drop=True)
    print("\n2. Test Set Final Features:")
    analyze_features(df_test, final_test)
    final_test.to_csv(os.path.join(out_folder, "test_set.csv"), index=False)
    
    if df_cons is not None:
        final_cons, removed_cons = generate_features(df_cons)
        final_cons["concept:name"] = final_cons["concept:name"].str.replace(r"^O_Sent\s*\(.*\)", "O_Sent", regex=True)
        final_cons = final_cons.sort_values(["case:concept:name", "occurrence_id"], ascending=[True, True]).reset_index(drop=True)
        print("\n3. Calibration Set Final Features:")
        analyze_features(df_cons, final_cons)
        final_cons.to_csv(os.path.join(out_folder, "calibration_set.csv"), index=False)
    
    print("\nDataset Pre-processing and Feature Generation completed!")

###############################################################################
# Example usage
###############################################################################
if __name__ == "__main__":
    REMOVE_MODE = True
    print("\n=== BPI Challenge 2017 Dataset Pre-Processing ===")
    xes_file = "dataset/BPI Challenge 2017.xes"
    log = xes_importer.apply(xes_file)
    df = converter.apply(log, variant=converter.Variants.TO_DATA_FRAME)

    start_date     = "2016-01"
    end_date       = "2017-01"
    max_days       = 47.81
    test_len_share = 0.2
    calibration_share = 0.2
    output_type    = "csv"
    out_path       = "."
    file_name      = ""

    remainTimeOrClassifBenchmark(
        dataset=df,
        path=out_path,
        file_name=file_name,
        start_date=start_date,
        end_date=end_date,
        max_days=max_days,
        test_len_share=test_len_share,
        output_type=output_type,
        keywords_dict=None,
        remove_missing_prefix_cases=REMOVE_MODE,
        calibration_share=calibration_share,
        apply_calibration=True
    )